# Clean Accuracy Evaluation — Multi-Model

Evaluates 3 models on the unperturbed LIAR test set. Results are saved under:
```
RESULTS/<model_name>/clean_accuracy_summary.json
RESULTS/<model_name>/clean_per_class_results.csv
RESULTS/<model_name>/clean_confusion_matrix.png
RESULTS/<model_name>/clean_per_class_f1.png
```

## 1. Install & Import

In [2]:
import os
import json
import random
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import seaborn as sns

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
    set_seed,
)
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    classification_report,
    confusion_matrix,
    precision_recall_fscore_support,
)

print("✅ All libraries imported")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")


✅ All libraries imported
PyTorch: 2.5.1+cu121
CUDA available: True
GPU: NVIDIA GeForce RTX 4070


## 2. Global Configuration
Edit `MODEL_DIRS` to point to your 3 model directories.

In [9]:
# ── Edit these ──────────────────────────────────────────────────────────────
MODEL_DIRS = [
    './baseline_42',
    'adversarial_trained_model_42.25/final_model',
    'adversarial_trained_model_42.50/final_model2',
      './baseline_123',
    'adversarial_trained_model_123.25/final_model3',
    'adversarial_trained_model_123.50/final_model4',
      './baseline_2026',
    'adversarial_trained_model_2026.25/final_model5',
    'adversarial_trained_model_2026.50/final_model6',
]
# ────────────────────────────────────────────────────────────────────────────

RESULTS_ROOT = 'RESULTS'   # same top-level folder as attack success rate
MAX_LENGTH   = 256
BATCH_SIZE   = 16
SEED         = 2026
# Label mapping  (must match training order)
LABELS   = ['false', 'half-true', 'mostly-true', 'true', 'barely-true', 'pants-fire']
label2id = {l: i for i, l in enumerate(LABELS)}
id2label = {i: l for l, i in label2id.items()}

def set_all_seeds(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    set_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False

set_all_seeds(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {device}')
print(f'Models : {MODEL_DIRS}')
print(f'Output : {RESULTS_ROOT}/<model_name>/')

Device : cuda
Models : ['./baseline_42', 'adversarial_trained_model_42.25/final_model', 'adversarial_trained_model_42.50/final_model2', './baseline_123', 'adversarial_trained_model_123.25/final_model3', 'adversarial_trained_model_123.50/final_model4', './baseline_2026', 'adversarial_trained_model_2026.25/final_model5', 'adversarial_trained_model_2026.50/final_model6']
Output : RESULTS/<model_name>/


## 3. Load & Preprocess Dataset (once)

In [4]:
print('Loading LIAR dataset...')
ds = load_dataset('ucsbnlp/liar', trust_remote_code=True)
print({k: len(ds[k]) for k in ds.keys()})

Loading LIAR dataset...
{'train': 10269, 'test': 1283, 'validation': 1284}


In [5]:
def safe_str(x):
    if x is None:
        return ''
    if isinstance(x, (list, tuple)):
        x = ', '.join([str(i) for i in x if i is not None])
    return str(x).strip()

def build_text(example):
    statement = safe_str(example.get('statement', ''))
    subject   = safe_str(example.get('subjects', example.get('subject', '')))
    context   = safe_str(example.get('context', ''))
    parts     = [statement]
    if subject: parts.append(f'SUBJECT: {subject}')
    if context: parts.append(f'CONTEXT: {context}')
    example['text'] = ' [SEP] '.join(parts)
    y = example.get('label')
    example['label_id'] = label2id[y.strip().lower()] if isinstance(y, str) else int(y)
    return example

ds = ds.map(build_text)
print(f"Example: {ds['test'][0]['text'][:150]}...")

Example: Building a wall on the U.S.-Mexico border will take literally years. [SEP] SUBJECT: immigration [SEP] CONTEXT: Radio interview...


## 4. Helper Functions

In [6]:
def get_out_dir(model_dir):
    model_name = os.path.basename(model_dir.rstrip('/\\'))
    out_dir    = os.path.join(RESULTS_ROOT, model_name)
    os.makedirs(out_dir, exist_ok=True)
    return out_dir, model_name

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions    = np.argmax(logits, axis=-1)
    return {
        'accuracy'         : accuracy_score(labels, predictions),
        'f1_macro'         : f1_score(labels, predictions, average='macro'),
        'f1_weighted'      : f1_score(labels, predictions, average='weighted'),
        'precision_macro'  : precision_score(labels, predictions, average='macro'),
        'recall_macro'     : recall_score(labels, predictions, average='macro'),
    }

def tokenize_fn(tokenizer, batch):
    return tokenizer(batch['text'], truncation=True, max_length=MAX_LENGTH, padding=False)

def save_confusion_matrix(true_labels, pred_labels, out_dir, model_name):
    cm = confusion_matrix(true_labels, pred_labels)
    plt.figure(figsize=(10, 8))
    sns.heatmap(
        cm, annot=True, fmt='d', cmap='Blues',
        xticklabels=LABELS, yticklabels=LABELS,
        cbar_kws={'label': 'Count'}
    )
    plt.xlabel('Predicted Label', fontsize=12)
    plt.ylabel('True Label', fontsize=12)
    plt.title(f'Confusion Matrix — {model_name}', fontsize=14, fontweight='bold')
    plt.xticks(rotation=45, ha='right')
    plt.yticks(rotation=0)
    plt.tight_layout()
    path = os.path.join(out_dir, 'clean_confusion_matrix.png')
    plt.savefig(path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f'  Confusion matrix → {path}')

def save_per_class_chart(f1_scores, out_dir, model_name):
    plt.figure(figsize=(10, 6))
    plt.bar(LABELS, f1_scores, color='steelblue', alpha=0.8)
    plt.xlabel('Label', fontsize=12)
    plt.ylabel('F1 Score', fontsize=12)
    plt.title(f'Per-Class F1 Scores — {model_name}', fontsize=14, fontweight='bold')
    plt.xticks(rotation=45, ha='right')
    plt.ylim(0, 1)
    plt.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    path = os.path.join(out_dir, 'clean_per_class_f1.png')
    plt.savefig(path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f'  Per-class F1 chart → {path}')

print('Helpers defined.')

Helpers defined.


## 5. Evaluate Each Model

In [7]:
all_summaries = []

for model_dir in MODEL_DIRS:
    out_dir, model_name = get_out_dir(model_dir)

    print('=' * 60)
    print(f'MODEL : {model_name}')
    print(f'OUTPUT: {out_dir}')
    print('=' * 60)

    # ── Load tokenizer & model ───────────────────────────────────
    tokenizer = AutoTokenizer.from_pretrained(model_dir, use_fast=True)
    model     = AutoModelForSequenceClassification.from_pretrained(
        model_dir,
        num_labels=len(LABELS),
        id2label=id2label,
        label2id=label2id,
    )
    model.to(device)
    model.eval()
    print(f'  Loaded | num_labels: {model.config.num_labels} | params: {sum(p.numel() for p in model.parameters()):,}')

    # ── Tokenize dataset ─────────────────────────────────────────
    tok_ds = ds.map(lambda b: tokenize_fn(tokenizer, b), batched=True)
    tok_ds = tok_ds.remove_columns(
        [c for c in tok_ds['test'].column_names
         if c not in ('input_ids', 'attention_mask', 'label_id')]
    )
    tok_ds = tok_ds.rename_column('label_id', 'labels')

    # ── Trainer (eval only) ──────────────────────────────────────
    data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
    training_args = TrainingArguments(
        output_dir=os.path.join(out_dir, 'eval_tmp'),
        per_device_eval_batch_size=BATCH_SIZE,
        dataloader_num_workers=0,
        report_to='none',
    )
    trainer = Trainer(
        model=model,
        args=training_args,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
    )

    # ── Test set evaluation ──────────────────────────────────────
    print('  Evaluating on test set...')
    test_results = trainer.evaluate(tok_ds['test'])

    pred_out    = trainer.predict(tok_ds['test'])
    true_labels = pred_out.label_ids
    pred_labels = np.argmax(pred_out.predictions, axis=-1)

    # ── Validation set evaluation ────────────────────────────────
    print('  Evaluating on validation set...')
    val_results = trainer.evaluate(tok_ds['validation'])

    # ── Per-class metrics ────────────────────────────────────────
    precision, recall, f1, support = precision_recall_fscore_support(
        true_labels, pred_labels, labels=range(len(LABELS))
    )
    per_class_df = pd.DataFrame({
        'Label'    : LABELS,
        'Precision': precision,
        'Recall'   : recall,
        'F1-Score' : f1,
        'Support'  : support,
    })

    # ── Save artefacts ───────────────────────────────────────────
    save_confusion_matrix(true_labels, pred_labels, out_dir, model_name)
    save_per_class_chart(f1, out_dir, model_name)

    csv_path = os.path.join(out_dir, 'clean_per_class_results.csv')
    per_class_df.to_csv(csv_path, index=False)
    print(f'  Per-class CSV → {csv_path}')

    # ── Save summary JSON ────────────────────────────────────────
    summary = {
        'model'               : model_name,
        'model_dir'           : model_dir,
        'test_accuracy'       : round(test_results['eval_accuracy'] * 100, 4),
        'test_f1_macro'       : round(test_results['eval_f1_macro'], 4),
        'test_f1_weighted'    : round(test_results['eval_f1_weighted'], 4),
        'test_precision_macro': round(test_results['eval_precision_macro'], 4),
        'test_recall_macro'   : round(test_results['eval_recall_macro'], 4),
        'test_loss'           : round(test_results['eval_loss'], 4),
        'val_accuracy'        : round(val_results['eval_accuracy'] * 100, 4),
        'val_f1_macro'        : round(val_results['eval_f1_macro'], 4),
        'val_loss'            : round(val_results['eval_loss'], 4),
    }
    summary_path = os.path.join(out_dir, 'clean_accuracy_summary.json')
    with open(summary_path, 'w', encoding='utf-8') as f:
        json.dump(summary, f, indent=2)
    print(f'  Summary → {summary_path}')

    all_summaries.append(summary)

    print(f'\n  ✓ {model_name}')
    print(f'    Test  Accuracy : {summary["test_accuracy"]:.2f}%')
    print(f'    Test  F1 Macro : {summary["test_f1_macro"]:.4f}')
    print(f'    Val   Accuracy : {summary["val_accuracy"]:.2f}%')

    # Free GPU memory before next model
    del model, tokenizer, trainer, tok_ds
    import gc; gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    print()

print('=' * 60)
print('ALL MODELS COMPLETE')
print('=' * 60)

MODEL : baseline_2026
OUTPUT: RESULTS\baseline_2026
  Loaded | num_labels: 6 | params: 435,067,910


Map:   0%|          | 0/10269 [00:00<?, ? examples/s]

Map:   0%|          | 0/1283 [00:00<?, ? examples/s]

Map:   0%|          | 0/1284 [00:00<?, ? examples/s]

  Evaluating on test set...


...

Map:   0%|          | 0/1283 [00:00<?, ? examples/s]

Map:   0%|          | 0/1284 [00:00<?, ? examples/s]

  Evaluating on test set...


  Evaluating on validation set...


  Confusion matrix → RESULTS\final_model6\clean_confusion_matrix.png
  Per-class F1 chart → RESULTS\final_model6\clean_per_class_f1.png
  Per-class CSV → RESULTS\final_model6\clean_per_class_results.csv
  Summary → RESULTS\final_model6\clean_accuracy_summary.json

  ✓ final_model6
    Test  Accuracy : 30.87%
    Test  F1 Macro : 0.3169
    Val   Accuracy : 30.45%

ALL MODELS COMPLETE


## 6. Comparison Table

In [10]:
# Load summaries (works even after a fresh kernel restart)
loaded = []
for model_dir in MODEL_DIRS:
    model_name   = os.path.basename(model_dir.rstrip('/\\'))
    summary_path = os.path.join(RESULTS_ROOT, model_name, 'clean_accuracy_summary.json')
    if os.path.exists(summary_path):
        with open(summary_path, 'r', encoding='utf-8') as f:
            loaded.append(json.load(f))
    else:
        print(f'WARNING: summary not found for {model_name}')

cmp_df = pd.DataFrame(loaded)[[
    'model', 'test_accuracy', 'test_f1_macro', 'test_f1_weighted',
    'test_precision_macro', 'test_recall_macro', 'val_accuracy', 'val_f1_macro'
]]
cmp_df.columns = [
    'Model', 'Test Acc (%)', 'Test F1 Macro', 'Test F1 Weighted',
    'Test Prec Macro', 'Test Rec Macro', 'Val Acc (%)', 'Val F1 Macro'
]
display(cmp_df)

# Save comparison CSV
os.makedirs(RESULTS_ROOT, exist_ok=True)
csv_path = os.path.join(RESULTS_ROOT, 'clean_accuracy_comparison_all.csv')
cmp_df.to_csv(csv_path, index=False)
print(f'\nComparison saved → {csv_path}')

,Model,Test Acc (%),Test F1 Macro,Test F1 Weighted,Test Prec Macro,Test Rec Macro,Val Acc (%),Val F1 Macro
0,baseline_42,31.6446,0.3295,0.3134,0.3461,0.3275,30.6854,0.3162
1,final_model,30.0078,0.3145,0.2981,0.3254,0.3133,29.8287,0.3103
2,final_model2,28.9945,0.2948,0.2795,0.3160,0.2998,30.9190,0.3076
3,baseline_123,30.8652,0.3244,0.3018,0.3253,0.3389,30.2960,0.3141
4,final_model3,30.5534,0.3140,0.3024,0.3376,0.3079,29.5950,0.2931
5,final_model4,31.4887,0.3218,0.3066,0.3447,0.3214,30.6854,0.3069
6,baseline_2026,30.7872,0.3179,0.3023,0.3273,0.3236,30.9190,0.3158
7,final_model5,30.3975,0.3091,0.2979,0.3200,0.3134,30.1402,0.3040
8,final_model6,30.8652,0.3169,0.3077,0.3346,0.3101,30.4517,0.3092



Comparison saved → RESULTS\clean_accuracy_comparison_all.csv
